In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "460_build_gold_vw_fact_project_item_comparison.py"
RUN_ID = str(uuid.uuid4())

SOURCE_COMPARISON_TABLE = f"{catalog}.gold.fact_project_item_comparison"
SOURCE_ROLE_CONTEXT_TABLE = f"{catalog}.gold.fact_project_bid_role_context"
SOURCE_ITEM_BID_TABLE = f"{catalog}.gold.fact_project_item_bid"
SOURCE_ITEM_ESTIMATE_TABLE = f"{catalog}.gold.fact_project_item_estimate"
TARGET_VIEW = f"{catalog}.gold.vw_fact_project_item_comparison"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_fact_project_item_comparison
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    WITH rc AS (
        SELECT *
        FROM {SOURCE_ROLE_CONTEXT_TABLE}
    )

    , focus_bid AS (
        SELECT
              b.project_id
            , b.vendor_name_standardized
            , b.bid_item_sequence_number
            , b.item_specification_key
            , b.project_key
            , b.vendor_key
            , b.project_classification_key
            , b.county_key
            , b.project_name
            , b.project_number
            , b.state_project_number
            , b.control_section_job_csj
            , b.controlling_project_id_ccsj
            , b.project_type
            , b.project_classification
            , b.project_actual_let_date
            , b.project_estimated_let_date
            , b.county
            , b.location
            , b.district_division
            , b.highway
            , b.short_description
            , b.bid_code
            , b.bid_item_description
            , b.measurement_unit
            , b.bid_item_quantity
            , b.bid_item_unit_price_amount
            , b.extended_item_amount_calc
            , b.specification_code
            , b.specification_description
            , b.effective_specification_description
            , b.item_work_category
            , b.item_work_category_source
            , b.is_defaulted_work_category
        FROM {SOURCE_ITEM_BID_TABLE} b
    )

    , benchmark_bid AS (
        SELECT
              b.project_id
            , b.vendor_name_standardized
            , b.bid_item_sequence_number
            , b.item_specification_key
            , b.bid_code
            , b.bid_item_description
            , b.measurement_unit
            , b.bid_item_quantity
            , b.bid_item_unit_price_amount
            , b.extended_item_amount_calc
        FROM {SOURCE_ITEM_BID_TABLE} b
    )

    , winner_bid AS (
        SELECT
              b.project_id
            , b.vendor_name_standardized
            , b.bid_item_sequence_number
            , b.item_specification_key
            , b.bid_code
            , b.bid_item_description
            , b.measurement_unit
            , b.bid_item_quantity
            , b.bid_item_unit_price_amount
            , b.extended_item_amount_calc
        FROM {SOURCE_ITEM_BID_TABLE} b
    )

    , second_bid AS (
        SELECT
              b.project_id
            , b.vendor_name_standardized
            , b.bid_item_sequence_number
            , b.item_specification_key
            , b.bid_code
            , b.bid_item_description
            , b.measurement_unit
            , b.bid_item_quantity
            , b.bid_item_unit_price_amount
            , b.extended_item_amount_calc
        FROM {SOURCE_ITEM_BID_TABLE} b
    )

    , estimate_side AS (
        SELECT
              e.project_id
            , e.bid_item_sequence_number
            , e.item_specification_key
            , e.bid_code
            , e.bid_item_description
            , e.measurement_unit
            , e.bid_item_quantity
            , e.engineer_estimate_unit_price
            , e.extended_estimate_item_amount_calc
            , e.project_engineer_total
            , e.engineer_project_total_source
            , e.specification_code
            , e.specification_description
            , e.effective_specification_description
            , e.item_work_category
            , e.item_work_category_source
            , e.is_defaulted_work_category
        FROM {SOURCE_ITEM_ESTIMATE_TABLE} e
    )

    SELECT
          cmp.project_item_comparison_key                    AS `Project Item Comparison Key`
        , COALESCE(f.project_key, rc.project_key)            AS `Project Key`
        , rc.focus_vendor_key                                AS `Vendor Key`
        , cmp.item_specification_key                         AS `Item Specification Key`

        , rc.project_classification_key                      AS `Project Classification Key`
        , rc.county_key                                      AS `County Key`

        , COALESCE(
              CASE
                  WHEN f.specification_code IS NOT NULL
                   AND f.specification_description IS NOT NULL
                  THEN concat(CAST(f.specification_code AS STRING), ' - ', f.specification_description)
                  WHEN f.specification_code IS NOT NULL
                  THEN CAST(f.specification_code AS STRING)
                  ELSE f.specification_description
              END
            , CASE
                  WHEN e.specification_code IS NOT NULL
                   AND e.specification_description IS NOT NULL
                  THEN concat(CAST(e.specification_code AS STRING), ' - ', e.specification_description)
                  WHEN e.specification_code IS NOT NULL
                  THEN CAST(e.specification_code AS STRING)
                  ELSE e.specification_description
              END
          )                                                  AS `Specification`

        , md5(COALESCE(f.item_work_category, e.item_work_category, '')) AS `Item Work Category Key`

        , cmp.project_id                                     AS `Project ID`
        , rc.focus_vendor_name                               AS `Vendor Name`
        , rc.focus_vendor_name_standardized                  AS `Vendor Name Standardized`

        , COALESCE(f.project_name, rc.project_name)          AS `Project Name`
        , COALESCE(f.project_number, rc.project_number)      AS `Project Number`
        , COALESCE(f.state_project_number, rc.state_project_number) AS `State Project Number`
        , COALESCE(f.control_section_job_csj, rc.control_section_job_csj) AS `Control Section Job CSJ`
        , COALESCE(f.controlling_project_id_ccsj, rc.controlling_project_id_ccsj) AS `Controlling Project ID CCSJ`
        , COALESCE(f.project_type, rc.project_type)          AS `Project Type`
        , COALESCE(f.project_classification, rc.project_classification) AS `Project Classification`
        , CAST(COALESCE(f.project_actual_let_date, rc.project_actual_let_date) AS DATE) AS `Project Actual Let Date`
        , COALESCE(f.project_estimated_let_date, rc.project_estimated_let_date) AS `Project Estimated Let Date`
        , rc.project_limits_from                             AS `Project Limits From`
        , rc.project_limits_to                               AS `Project Limits To`
        , COALESCE(f.county, rc.county)                      AS `County`
        , COALESCE(f.location, rc.location)                  AS `Location`
        , COALESCE(f.district_division, rc.district_division) AS `District Division`
        , COALESCE(f.highway, rc.highway)                    AS `Highway`
        , COALESCE(f.short_description, rc.short_description) AS `Short Description`
        , rc.federal_project_number                          AS `Federal Project Number`

        , cmp.bid_item_sequence_number                       AS `Bid Item Sequence Number`

        , rc.focus_vendor_name                               AS `Focus Vendor Name`
        , rc.benchmark_vendor_name                           AS `Benchmark Vendor Name`
        , rc.winner_vendor_name                              AS `Winner Vendor Name`
        , rc.second_vendor_name                              AS `Second Vendor Name`
        , rc.benchmark_definition                            AS `Benchmark Definition`

        , f.bid_code                                         AS `Focus Bid Code`
        , bb.bid_code                                        AS `Benchmark Bid Code`
        , wb.bid_code                                        AS `Winner Bid Code`
        , sb.bid_code                                        AS `Second Bid Code`
        , e.bid_code                                         AS `Estimate Bid Code`

        , f.bid_item_description                             AS `Focus Item Description`
        , bb.bid_item_description                            AS `Benchmark Item Description`
        , wb.bid_item_description                            AS `Winner Item Description`
        , sb.bid_item_description                            AS `Second Item Description`
        , e.bid_item_description                             AS `Estimate Item Description`

        , f.measurement_unit                                 AS `Focus Measurement Unit`
        , bb.measurement_unit                                AS `Benchmark Measurement Unit`
        , wb.measurement_unit                                AS `Winner Measurement Unit`
        , sb.measurement_unit                                AS `Second Measurement Unit`
        , e.measurement_unit                                 AS `Estimate Measurement Unit`
        , COALESCE(f.measurement_unit, bb.measurement_unit, wb.measurement_unit, sb.measurement_unit, e.measurement_unit) AS `Comparison Measurement Unit`

        , f.bid_item_quantity                                AS `Focus Quantity`
        , bb.bid_item_quantity                               AS `Benchmark Quantity`
        , wb.bid_item_quantity                               AS `Winner Quantity`
        , sb.bid_item_quantity                               AS `Second Quantity`
        , e.bid_item_quantity                                AS `Estimate Quantity`
        , COALESCE(f.bid_item_quantity, bb.bid_item_quantity, wb.bid_item_quantity, sb.bid_item_quantity, e.bid_item_quantity) AS `Comparison Quantity`

        , f.bid_item_unit_price_amount                       AS `Focus Unit Price`
        , bb.bid_item_unit_price_amount                      AS `Benchmark Unit Price`
        , wb.bid_item_unit_price_amount                      AS `Winner Unit Price`
        , sb.bid_item_unit_price_amount                      AS `Second Unit Price`
        , e.engineer_estimate_unit_price                     AS `Estimate Unit Price`

        , f.extended_item_amount_calc                        AS `Focus Extended Price`
        , bb.extended_item_amount_calc                       AS `Benchmark Extended Price`
        , wb.extended_item_amount_calc                       AS `Winner Extended Price`
        , sb.extended_item_amount_calc                       AS `Second Extended Price`
        , e.extended_estimate_item_amount_calc               AS `Estimate Extended Price`

        , cmp.focus_vs_benchmark_amount                      AS `Focus Vs Benchmark Amount`
        , ABS(cmp.focus_vs_benchmark_amount)                 AS `ABS Focus Vs Benchmark Amount`
        , cmp.focus_vs_benchmark_ratio                       AS `Focus Vs Benchmark Ratio`
        , CASE WHEN cmp.focus_vs_benchmark_amount < 0 THEN TRUE ELSE FALSE END AS `Focus Under Benchmark Flag`
        , CASE WHEN cmp.focus_vs_benchmark_amount > 0 THEN TRUE ELSE FALSE END AS `Focus Over Benchmark Flag`
        , CASE WHEN cmp.focus_vs_benchmark_amount = 0 THEN TRUE ELSE FALSE END AS `Focus Benchmark Tie Flag`

        , CASE
              WHEN wb.extended_item_amount_calc IS NOT NULL
               AND sb.extended_item_amount_calc IS NOT NULL
              THEN wb.extended_item_amount_calc - sb.extended_item_amount_calc
          END                                               AS `Winner Vs Second Amount`
        , CASE
              WHEN wb.extended_item_amount_calc IS NOT NULL
               AND sb.extended_item_amount_calc IS NOT NULL
              THEN ABS(wb.extended_item_amount_calc - sb.extended_item_amount_calc)
          END                                               AS `ABS Winner Vs Second Amount`
        , CASE
              WHEN sb.extended_item_amount_calc IS NOT NULL
               AND sb.extended_item_amount_calc <> 0
               AND wb.extended_item_amount_calc IS NOT NULL
              THEN wb.extended_item_amount_calc / sb.extended_item_amount_calc
          END                                               AS `Winner Vs Second Ratio`
        , CASE
              WHEN wb.extended_item_amount_calc IS NOT NULL
               AND sb.extended_item_amount_calc IS NOT NULL
               AND wb.extended_item_amount_calc < sb.extended_item_amount_calc
              THEN TRUE ELSE FALSE
          END                                               AS `Winner Under Second Flag`
        , CASE
              WHEN wb.extended_item_amount_calc IS NOT NULL
               AND sb.extended_item_amount_calc IS NOT NULL
               AND wb.extended_item_amount_calc > sb.extended_item_amount_calc
              THEN TRUE ELSE FALSE
          END                                               AS `Winner Over Second Flag`
        , CASE
              WHEN wb.extended_item_amount_calc IS NOT NULL
               AND sb.extended_item_amount_calc IS NOT NULL
               AND wb.extended_item_amount_calc = sb.extended_item_amount_calc
              THEN TRUE ELSE FALSE
          END                                               AS `Winner Second Item Tie Flag`

        , cmp.focus_vs_estimate_amount                      AS `Focus Vs Estimate Amount`
        , CASE
              WHEN f.bid_item_unit_price_amount IS NOT NULL
               AND e.engineer_estimate_unit_price IS NOT NULL
              THEN f.bid_item_unit_price_amount - e.engineer_estimate_unit_price
          END                                               AS `Focus Vs Estimate Unit Price Diff`
        , CASE
              WHEN e.extended_estimate_item_amount_calc IS NOT NULL
               AND e.extended_estimate_item_amount_calc <> 0
               AND f.extended_item_amount_calc IS NOT NULL
              THEN f.extended_item_amount_calc / e.extended_estimate_item_amount_calc
          END                                               AS `Focus Vs Estimate Amount Ratio`
        , CASE
              WHEN e.engineer_estimate_unit_price IS NOT NULL
               AND e.engineer_estimate_unit_price <> 0
               AND f.bid_item_unit_price_amount IS NOT NULL
              THEN f.bid_item_unit_price_amount / e.engineer_estimate_unit_price
          END                                               AS `Focus Vs Estimate Unit Price Ratio`

        , COALESCE(f.specification_code, e.specification_code) AS `Specification Code`
        , COALESCE(f.specification_description, e.specification_description) AS `Specification Description`
        , COALESCE(f.effective_specification_description, e.effective_specification_description) AS `Effective Specification Description`
        , COALESCE(f.item_work_category, e.item_work_category) AS `Item Work Category`
        , COALESCE(f.item_work_category_source, e.item_work_category_source) AS `Item Work Category Source`
        , COALESCE(f.is_defaulted_work_category, e.is_defaulted_work_category) AS `Is Defaulted Work Category`

        , CASE WHEN f.project_id IS NOT NULL THEN TRUE ELSE FALSE END AS `Has Focus Row`
        , CASE WHEN bb.project_id IS NOT NULL THEN TRUE ELSE FALSE END AS `Has Benchmark Row`
        , CASE WHEN wb.project_id IS NOT NULL THEN TRUE ELSE FALSE END AS `Has Winner Row`
        , CASE WHEN sb.project_id IS NOT NULL THEN TRUE ELSE FALSE END AS `Has Second Row`
        , CASE WHEN e.project_id IS NOT NULL THEN TRUE ELSE FALSE END AS `Has Estimate Row`
        , CASE
              WHEN f.project_id IS NOT NULL
                OR bb.project_id IS NOT NULL
                OR wb.project_id IS NOT NULL
                OR sb.project_id IS NOT NULL
              THEN TRUE ELSE FALSE
          END                                               AS `Has Bid Row`
        , CASE
              WHEN (
                   f.project_id IS NOT NULL
                OR bb.project_id IS NOT NULL
                OR wb.project_id IS NOT NULL
                OR sb.project_id IS NOT NULL
              )
               AND e.project_id IS NOT NULL
              THEN TRUE ELSE FALSE
          END                                               AS `Is Matched Item`
        , CASE
              WHEN (
                   f.project_id IS NULL
               AND bb.project_id IS NULL
               AND wb.project_id IS NULL
               AND sb.project_id IS NULL
              )
               AND e.project_id IS NOT NULL
              THEN TRUE ELSE FALSE
          END                                               AS `Estimate Only Item`
        , CASE
              WHEN (
                   f.project_id IS NOT NULL
                OR bb.project_id IS NOT NULL
                OR wb.project_id IS NOT NULL
                OR sb.project_id IS NOT NULL
              )
               AND e.project_id IS NULL
              THEN TRUE ELSE FALSE
          END                                               AS `Bid Only Item`

        , CASE
              WHEN f.measurement_unit IS NOT NULL
               AND bb.measurement_unit IS NOT NULL
               AND f.measurement_unit = bb.measurement_unit
              THEN TRUE ELSE FALSE
          END                                               AS `Focus Benchmark Measurement Unit Matches`
        , CASE
              WHEN wb.measurement_unit IS NOT NULL
               AND sb.measurement_unit IS NOT NULL
               AND wb.measurement_unit = sb.measurement_unit
              THEN TRUE ELSE FALSE
          END                                               AS `Winner Second Measurement Unit Matches`
        , CASE
              WHEN f.bid_item_quantity IS NOT NULL
               AND bb.bid_item_quantity IS NOT NULL
               AND f.bid_item_quantity = bb.bid_item_quantity
              THEN TRUE ELSE FALSE
          END                                               AS `Focus Benchmark Quantity Matches`
        , CASE
              WHEN wb.bid_item_quantity IS NOT NULL
               AND sb.bid_item_quantity IS NOT NULL
               AND wb.bid_item_quantity = sb.bid_item_quantity
              THEN TRUE ELSE FALSE
          END                                               AS `Winner Second Quantity Matches`
        , CASE
              WHEN f.bid_code IS NOT NULL
               AND bb.bid_code IS NOT NULL
               AND f.bid_code = bb.bid_code
              THEN TRUE ELSE FALSE
          END                                               AS `Focus Benchmark Bid Code Matches`
        , CASE
              WHEN wb.bid_code IS NOT NULL
               AND sb.bid_code IS NOT NULL
               AND wb.bid_code = sb.bid_code
              THEN TRUE ELSE FALSE
          END                                               AS `Winner Second Bid Code Matches`
        , CASE
              WHEN COALESCE(f.bid_code, bb.bid_code, wb.bid_code, sb.bid_code) IS NOT NULL
               AND e.bid_code IS NOT NULL
               AND COALESCE(f.bid_code, bb.bid_code, wb.bid_code, sb.bid_code) = e.bid_code
              THEN TRUE ELSE FALSE
          END                                               AS `Bid Code Matches`
        , CASE
              WHEN COALESCE(f.measurement_unit, bb.measurement_unit, wb.measurement_unit, sb.measurement_unit) IS NOT NULL
               AND e.measurement_unit IS NOT NULL
               AND COALESCE(f.measurement_unit, bb.measurement_unit, wb.measurement_unit, sb.measurement_unit) = e.measurement_unit
              THEN TRUE ELSE FALSE
          END                                               AS `Measurement Unit Matches`
        , CASE
              WHEN COALESCE(f.bid_item_quantity, bb.bid_item_quantity, wb.bid_item_quantity, sb.bid_item_quantity) IS NOT NULL
               AND e.bid_item_quantity IS NOT NULL
               AND COALESCE(f.bid_item_quantity, bb.bid_item_quantity, wb.bid_item_quantity, sb.bid_item_quantity) = e.bid_item_quantity
              THEN TRUE ELSE FALSE
          END                                               AS `Quantity Matches`
        , CASE
              WHEN COALESCE(f.extended_item_amount_calc, bb.extended_item_amount_calc, wb.extended_item_amount_calc, sb.extended_item_amount_calc) IS NOT NULL
               AND e.extended_estimate_item_amount_calc IS NOT NULL
              THEN COALESCE(f.extended_item_amount_calc, bb.extended_item_amount_calc, wb.extended_item_amount_calc, sb.extended_item_amount_calc)
                   - e.extended_estimate_item_amount_calc
          END                                               AS `Bid Vs Estimate Amount Diff`
        , CASE
              WHEN COALESCE(f.bid_item_unit_price_amount, bb.bid_item_unit_price_amount, wb.bid_item_unit_price_amount, sb.bid_item_unit_price_amount) IS NOT NULL
               AND e.engineer_estimate_unit_price IS NOT NULL
              THEN COALESCE(f.bid_item_unit_price_amount, bb.bid_item_unit_price_amount, wb.bid_item_unit_price_amount, sb.bid_item_unit_price_amount)
                   - e.engineer_estimate_unit_price
          END                                               AS `Bid Vs Estimate Unit Price Diff`
        , CASE
              WHEN e.extended_estimate_item_amount_calc IS NOT NULL
               AND e.extended_estimate_item_amount_calc <> 0
               AND COALESCE(f.extended_item_amount_calc, bb.extended_item_amount_calc, wb.extended_item_amount_calc, sb.extended_item_amount_calc) IS NOT NULL
              THEN COALESCE(f.extended_item_amount_calc, bb.extended_item_amount_calc, wb.extended_item_amount_calc, sb.extended_item_amount_calc)
                   / e.extended_estimate_item_amount_calc
          END                                               AS `Bid Vs Estimate Amount Ratio`
        , CASE
              WHEN e.engineer_estimate_unit_price IS NOT NULL
               AND e.engineer_estimate_unit_price <> 0
               AND COALESCE(f.bid_item_unit_price_amount, bb.bid_item_unit_price_amount, wb.bid_item_unit_price_amount, sb.bid_item_unit_price_amount) IS NOT NULL
              THEN COALESCE(f.bid_item_unit_price_amount, bb.bid_item_unit_price_amount, wb.bid_item_unit_price_amount, sb.bid_item_unit_price_amount)
                   / e.engineer_estimate_unit_price
          END                                               AS `Bid Vs Estimate Unit Price Ratio`

        , rc.bidder_count_in_project                        AS `Bidder Count In Project`
        , rc.lowest_bid_amount_in_project                   AS `Lowest Bid Amount In Project`
        , rc.highest_bid_amount_in_project                  AS `Highest Bid Amount In Project`
        , rc.bid_spread_amount_in_project                   AS `Bid Spread Amount In Project`
        , rc.focus_bid_total_amount                         AS `Focus Bid Total Amount`
        , rc.benchmark_bid_total_amount                     AS `Benchmark Bid Total Amount`
        , rc.winner_bid_total_amount                        AS `Winner Bid Total Amount`
        , rc.second_bid_total_amount                        AS `Second Bid Total Amount`
        , rc.estimate_project_total                         AS `Project Engineer Total`
        , 'PROJECT_ROLE_CONTEXT'                            AS `Engineer Project Total Source`

        , cmp.over_under_flag                               AS `Over Under Flag`
        , COALESCE(f.item_work_category, e.item_work_category) AS `Item Analysis Category`
        , COALESCE(f.item_work_category, e.item_work_category) AS `Analysis Category Group`

    FROM {SOURCE_COMPARISON_TABLE} cmp
    LEFT JOIN rc
        ON cmp.project_id = rc.project_id
    LEFT JOIN focus_bid f
        ON cmp.project_id = f.project_id
       AND rc.focus_vendor_name_standardized = f.vendor_name_standardized
       AND cmp.bid_item_sequence_number = f.bid_item_sequence_number
       AND cmp.item_specification_key = f.item_specification_key
    LEFT JOIN benchmark_bid bb
        ON cmp.project_id = bb.project_id
       AND rc.benchmark_vendor_name_standardized = bb.vendor_name_standardized
       AND cmp.bid_item_sequence_number = bb.bid_item_sequence_number
       AND cmp.item_specification_key = bb.item_specification_key
    LEFT JOIN winner_bid wb
        ON cmp.project_id = wb.project_id
       AND rc.winner_vendor_name_standardized = wb.vendor_name_standardized
       AND cmp.bid_item_sequence_number = wb.bid_item_sequence_number
       AND cmp.item_specification_key = wb.item_specification_key
    LEFT JOIN second_bid sb
        ON cmp.project_id = sb.project_id
       AND rc.second_vendor_name_standardized = sb.vendor_name_standardized
       AND cmp.bid_item_sequence_number = sb.bid_item_sequence_number
       AND cmp.item_specification_key = sb.item_specification_key
    LEFT JOIN estimate_side e
        ON cmp.project_id = e.project_id
       AND cmp.bid_item_sequence_number = e.bid_item_sequence_number
       AND cmp.item_specification_key = e.item_specification_key
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_VIEW}").collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_fact_project_item_comparison successfully.")

    print("=" * 70)
    print("Built gold.vw_fact_project_item_comparison")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise